# Notebook 01 — Exploration du jeu de données

**Projet :** Prédicteur de Rentabilité d'Investissement  
**Objectif :** Comprendre la structure du jeu de données, les distributions,
les corrélations et la qualité des données avant toute modélisation.

---

## Sommaire
1. Chargement et vue d'ensemble
2. Distribution de la variable cible
3. Statistiques descriptives
4. Distributions des variables continues
5. Analyse des variables catégorielles
6. Corrélations
7. Valeurs manquantes
8. Conclusions

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from config import DATA_DIR

# Style graphique
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
})
sns.set_palette('muted')

print('Librairies chargées ✅')

## 1. Chargement et vue d'ensemble

In [ ]:
# Chargement du jeu de données brut (généré par main.py)
chemin = DATA_DIR / 'investissements_brut.csv'

if not chemin.exists():
    # Génération à la volée si nécessaire
    from data import _generer_donnees_brutes
    df = _generer_donnees_brutes(n=20_000, graine=42)
    df.to_csv(chemin, index=False)
    print('Données générées ✅')
else:
    df = pd.read_csv(chemin)
    print('Données chargées ✅')

print(f'Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
df.head()

In [ ]:
# Types de données et mémoire utilisée
df.info()

## 2. Distribution de la variable cible

In [ ]:
taux = df['rentable'].value_counts(normalize=True)
print('Distribution de la variable cible :')
print(f"  Rentable     (1) : {taux.get(1, 0):.1%}")
print(f"  Non rentable (0) : {taux.get(0, 0):.1%}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Graphique en barres
counts = df['rentable'].value_counts().sort_index()
axes[0].bar(['Non rentable', 'Rentable'], counts.values,
            color=['#C62828', '#2E7D32'], width=0.45)
axes[0].set_title('Répartition absolue')
axes[0].set_ylabel('Nombre de projets')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

# Camembert
axes[1].pie(counts.values, labels=['Non rentable', 'Rentable'],
            autopct='%1.1f%%', colors=['#EF9A9A', '#A5D6A7'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Répartition relative')

plt.tight_layout()
plt.show()

## 3. Statistiques descriptives

In [ ]:
df.select_dtypes(include='number').describe().round(4).T.style.background_gradient(cmap='Blues', axis=0)

## 4. Distributions des variables continues

In [ ]:
variables_continues = [
    'roi_brut_annuel', 'score_risque', 'horizon_annees',
    'levier_financier', 'taux_inflation', 'liquidite',
    'diversification', 'experience_annees'
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, var in enumerate(variables_continues):
    for val, couleur in [(0, '#EF9A9A'), (1, '#A5D6A7')]:
        axes[i].hist(
            df.loc[df['rentable'] == val, var],
            bins=35, alpha=0.65, color=couleur, edgecolor='none',
            label='Non rentable' if val == 0 else 'Rentable'
        )
    axes[i].set_title(var)
    axes[i].tick_params(labelsize=8)

axes[0].legend(fontsize=8)
plt.suptitle('Distributions des variables continues par résultat', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Analyse des variables catégorielles

In [ ]:
cat_cols = ['secteur', 'zone_geo', 'type_actif', 'notation_credit']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    taux_rentable = df.groupby(col)['rentable'].mean().sort_values(ascending=False)
    barres = axes[i].bar(taux_rentable.index, taux_rentable.values * 100,
                         color='#1565C0', edgecolor='none')
    for b in barres:
        axes[i].text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
                     f'{b.get_height():.1f}%', ha='center', fontsize=8)
    axes[i].set_title(f'Taux de rentabilité par {col}')
    axes[i].set_ylabel('Taux de rentabilité (%)')
    axes[i].set_ylim(0, 100)
    axes[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 6. Corrélations

In [ ]:
# Corrélation de chaque variable numérique avec la cible
corr_cible = df.select_dtypes(include='number').corr()['rentable'].drop('rentable').sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
couleurs = ['#C62828' if v < 0 else '#2E7D32' for v in corr_cible]
ax.barh(corr_cible.index, corr_cible.values, color=couleurs, edgecolor='none')
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_title('Corrélation de Pearson avec la variable cible (rentable)', pad=10)
ax.set_xlabel('Corrélation')
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation complète
num_cols = df.select_dtypes(include='number').columns.tolist()
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    df[num_cols].corr(),
    annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Matrice de corrélation', pad=12)
plt.tight_layout()
plt.show()

## 7. Valeurs manquantes

In [ ]:
manquantes = df.isnull().sum()
if manquantes.sum() == 0:
    print('✅ Aucune valeur manquante dans le jeu de données.')
else:
    print('Valeurs manquantes par colonne :')
    print(manquantes[manquantes > 0])

## 8. Conclusions

**Points clés identifiés :**

- Le ROI brut annuel est le prédicteur le plus fortement corrélé avec la rentabilité.
- Le score de risque et le levier financier sont négativement corrélés avec la cible.
- Les secteurs *technologie* et *santé* présentent des taux de rentabilité supérieurs à la moyenne.
- Les marchés émergents et l'Europe de l'Est affichent des taux plus faibles.
- Les notations de crédit élevées (AAA, AA) sont associées à une meilleure rentabilité.
- Aucune valeur manquante — le jeu de données est prêt pour la modélisation.

→ Passer au **Notebook 02** pour l'ingénierie des variables et la modélisation.